In [ ]:
import os
import yaml
from pathlib import Path
from ultralytics import YOLO
import torch
from datetime import datetime
from PIL import Image
from tqdm import tqdm
import os
import numpy as np
import cv2

In [ ]:
# Load mô hình với pretrained
model = YOLO(model= "model/obb/yolo11n-obb.pt")



# Train the model với cấu hình GPU
results = model.train(
    data="data/yolov8_obb_dataset/dataset.yaml",
    epochs=250,
    imgsz=640,
    device=0,  # Sử dụng GPU đầu tiên (0), hoặc 'cpu' nếu muốn dùng CPU
    batch=32,  # Điều chỉnh batch size phù hợp với GPU memory
    workers=4,  # Số worker threads
    verbose=True,
    amp=False,
    # amp_file="model/obb/yolo11n.pt"

)
# model.amp_file="model/obb/yolo11n.pt"

In [ ]:
model = YOLO("runs/obb/train_n_latest/weights/best.pt")  # load a custom model

In [ ]:
# Validate the model
metrics = model.val()  # no arguments needed, dataset and settings remembered
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps  # a list contains map50-95 of each category

In [ ]:
# Run batched inference on a list of images
results = model.predict(
    ["/home/tuanndq/ocr_cccd/yolo/data/dataset_obb_v1_labeled/294_1_VTM_20240925103911_681.jpg",
    "/home/tuanndq/ocr_cccd/yolo/data/dataset_obb_v1_labeled/928_3_VTM_20241217190355_144.jpg",
    "data/dataset_obb_v1_labeled/Android_MYMBF_20250217_091720.020_1739758802478.jpg",
    "data/dataset_obb_v1_labeled/EKYC_Android_20250321_011435.035_Zbt_obl.jpg",
    "data/yolov8_obb_dataset/train/images/602_1_VTM_20240808103312_409.jpg",
    "data/dataset_obb_v1_labeled/UPDATE_LICENSE_20240701_161846.575_ci4_ofl.jpg",
    ],
    imgsz = 640,
    conf = 0.1,
    stream=True
)  # return a generator of Results objects

# Process results generator
for i,result in enumerate(results):
    boxes = result.boxes  # Boxes object for bounding box outputs
    masks = result.masks  # Masks object for segmentation masks outputs
    keypoints = result.keypoints  # Keypoints object for pose outputs
    probs = result.probs  # Probs object for classification outputs
    obb = result.obb  # Oriented boxes object for OBB outputs
    # result.show()  # display to screen
    result.save(filename=f"result{i}.jpg")  # save to disk

# CROP CARD

In [ ]:
from PIL import Image
import numpy as np
import cv2

In [ ]:
def _crop_and_rotate_image(image: np.ndarray, box: np.ndarray, expand_factor: float = 1.05):
        """
        Crop and rotate the image based on OBB box with customizable expansion.

        Args:
            image: Input image as numpy array
            box: OBB box [x_center, y_center, width, height, angle]
            expand_factor: Factor to expand the bounding box size (default 1.0 for original size)

        Returns:
            Cropped and rotated image as numpy array, or None if failed

        Nới rộng box theo tỷ lệ chiều rộng/dài
        """
        try:
            x_center, y_center, width, height, angle = box
            angle = angle * 180 / np.pi  # Convert radians to degrees
            # if width >= height:
            #     long_side = width
            #     short_side = height
            # else:
            #     long_side = height
            #     short_side = width

            # h_img, w_img, _ = image.shape
            # Apply expansion factor to width and height
            expanded_width = width * expand_factor
            expanded_height = height * expand_factor
            # if w_img/h_img>=0.5:
            #     expanded_width = expanded_width + expanded_width*0.2

            # Get rotation matrix
            M = cv2.getRotationMatrix2D((x_center, y_center), angle, 1.0)

            # Rotate the image
            rotated = cv2.warpAffine(image, M, (image.shape[1], image.shape[0]))

            # Calculate new coordinates with expanded dimensions
            x1 = int(x_center - expanded_width / 2)
            y1 = int(y_center - expanded_height / 2)
            x2 = int(x_center + expanded_width / 2)
            y2 = int(y_center + expanded_height / 2)

            # Ensure coordinates are within image bounds
            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(image.shape[1], x2)
            y2 = min(image.shape[0], y2)

            cropped = rotated[y1:y2, x1:x2]

            # Check if cropped image is valid
            if cropped.size == 0:
                return None

            return cropped
        except Exception as e:
            print(f"Error in cropping/rotating image: {str(e)}")
            return None

def crop_obb_images(image_folder, output_folder):
    model = YOLO("runs/obb/train_n_latest/weights/best.pt")
    classes_all = model.names

    for img_name in tqdm(os.listdir(image_folder)):
        if img_name.endswith(('.jpg', '.jpeg', '.png')):
            try:
                img_path = os.path.join(image_folder, img_name)
                # print(img_path)
                img = cv2.imread(img_path)
                results = model.predict(img_path, save=False, conf=0.5, imgsz=640)

                obb = results[0].obb # OBB object, only one img , so index = 0
                boxes = obb.xywhr.cpu().numpy()  # Hộp giới hạn OBB
                scores = obb.conf.cpu().numpy()  # Độ tin cậy
                classes = obb.cls.cpu().numpy().astype(int)  # Lớp

                # Chọn hộp có độ tin cậy cao nhất
                max_conf_idx = scores.argmax()
                best_box = boxes[max_conf_idx]
                best_class = classes[max_conf_idx]
                cls_name = classes_all[int(best_class)]
                best_score = scores[max_conf_idx]
                # print(cls_name)
                img_crop = _crop_and_rotate_image(img, best_box)
                # print(img_crop.shape)
                out_img_path = os.path.join(output_folder,cls_name)
                # print(out_img_path)

                if not os.path.exists(out_img_path):
                    os.makedirs(out_img_path)
                cv2.imwrite(os.path.join(out_img_path,img_name), img_crop)
            except:
                print("error image")

In [ ]:
image_folder =  "data/dataset_obb_v1_labeled"
output_folder = "data/dataset_cropped_card"
crop_obb_images(image_folder, output_folder)

In [ ]:
img_test = cv2.imread("data/dataset_obb_v1_labeled/UPDATE_LICENSE_Android_20240718_042535.035_VWE_ofl.jpg")
print(img_test.shape)